# Data Creation

Turns the raw monthly panel into a modelling snapshot: **one input read**, a set of column-creation steps (forward rollups for targets, backward rollups for engineered features), then **one output write**. It is deliberately separate from the modelling spine — with a proper feature store this logic would live there, not in a notebook.

All steps run through the shared helpers in `src/rollups.py`; no functions are defined in the product sections. Each further product is a new section (`## 2. FX Reactivation`, …) reusing the same helpers with different columns, windows, and thresholds.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession, Window, functions as F
spark = SparkSession.builder.getOrCreate()

from src.rollups import add_month_index, forward_rollup, backward_rollup, binary_target

## 1. FX Activation

### Read the panel — single input
One HDFS read into a Spark dataframe. The panel is one row per CIF per month. `add_month_index` adds a contiguous integer month so the rollup windows count in whole months.

In [ ]:
INPUT_PATH  = 'hdfs:///retail/monthly_panel'              # single input
OUTPUT_PATH = 'hdfs:///retail/snapshots/fx_activation'    # single output

df = spark.read.parquet(INPUT_PATH)
df = add_month_index(df, month_col='snapshot_month')
df.printSchema()

### Backward rollups — engineered feature columns
Aggregate activity over the *previous* months, anchored at each month. Example: FX transactions in the trailing 3 months.

In [ ]:
df = backward_rollup(df, id_col='cif', value_col='fx_txn', months=3,
                     out_col='fx_txn_count_3m', agg='sum')

### Forward rollup — the target column
Aggregate activity over the *next* months, then threshold. FX activation = at least one qualifying FX transaction in the 3 months after the observation month. Because the window is strictly forward, the label for month M is measured only over (M, M+3] — this is the time separation that keeps features and label from overlapping.

In [ ]:
df = forward_rollup(df, id_col='cif', value_col='fx_txn', months=3,
                    out_col='fx_fwd_txn_3m', agg='sum')
df = binary_target(df, signal_col='fx_fwd_txn_3m', threshold=1,
                   out_col='fx_activation_target')

### Guard the panel edge
The last months of the panel have no full 3-month forward window, so their target would rest on a partial window. Null those rows so they can't be used as training or OOT.

In [ ]:
w_cnt = Window.partitionBy('cif').orderBy('month_idx').rangeBetween(1, 3)
df = df.withColumn('_fwd_months', F.count('month_idx').over(w_cnt))
df = df.withColumn('fx_activation_target',
                   F.when(F.col('_fwd_months') >= 3, F.col('fx_activation_target')))
df = df.drop('_fwd_months')

### Sanity check
Eyeball the target rate at the observation month before writing. An implausibly high rate usually means eligibility wasn't applied or the forward signal leaked.

In [ ]:
(df.filter(F.col('snapshot_month') == '2026-03-01')
   .agg(F.avg('fx_activation_target').alias('base_rate'),
        F.count('*').alias('rows'))
   .show())

### Write — single output
One write to a single HDFS path. This path is what `configs/fx_activation.yaml` → `table` points to.

In [ ]:
df.write.mode('overwrite').parquet(OUTPUT_PATH)